# API Testing

In [1]:
import pandas as pd
import numpy as np
import time

import sys
import os

import requests
import json
from pathlib import Path, PurePath
from datetime import datetime

c:\Users\MULINGWA STEPHEN\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [2]:
# Token for API access
token = "338f417f5f8ae25ba6eb01c878134153FE905CC0A125F5E827810F937A192125F15C8769"

In [3]:
def wialon_login(token, full=False):
    url = "https://hst-api.wialon.com/wialon/ajax.html"

    params = {
        "svc": "token/login",
        "params": json.dumps({"token": token})
    }

    response = requests.post(url, params=params)
    result = response.json()

    if full:
        return result
    return result.get("eid")

In [4]:
eid = wialon_login(token)
print("EID:", eid)

EID: 10a02e2c38601c0c32b3c20029a4889f


In [5]:
# Get full formatted JSON
full_json = wialon_login(token, full=True)
print(json.dumps(full_json, indent=4))

{
    "host": "196.202.174.2",
    "eid": "10adadb8823822f81f7c7d27cf94832d",
    "gis_sid": "816104de161709d",
    "au": "musyoka@controltech-ea.com",
    "tm": 1779956522,
    "wsdk_version": "1.920.19",
    "base_url": "https://hst-api.wialon.com",
    "hw_gw_ip": "193.193.165.165",
    "hw_gw_dns": "nl.gpsgsm.org",
    "gis_search": "",
    "gis_render": "",
    "gis_geocode": "",
    "gis_routing": "",
    "billing_by_codes": "0",
    "drivers_feature_flags": "5",
    "feature_flags": "{\"notifications\":3}",
    "user": {
        "nm": "musyoka@controltech-ea.com",
        "cls": 1,
        "id": 30037778,
        "prp": {
            "__sensolator_resource_id": "17082202",
            "access_templates": "{\"avl_unit\":[],\"avl_unit_group\":[],\"avl_resource\":[],\"avl_route\":[],\"user\":[]}",
            "add_calculator": "13",
            "addr_provider": "map_google",
            "autoFillPromo": "1",
            "autocomplete": "{\"brand\":[\"FAW\",\"Scania\",\"Mercedes Ben

In [6]:
# Get full JSON
result = wialon_login(token, full=True)

# Save to file
with open("wialon_login.json", "w") as f:
    json.dump(result, f, indent=4)

print("JSON saved successfully!")

JSON saved successfully!


## Getting all the Resources

In [7]:
def get_resources(eid):
    url = "https://hst-api.wialon.com/wialon/ajax.html"
    params = {
        "svc": "core/search_items",
        "params": json.dumps({
            "spec": {"itemsType": "avl_resource", "propName": "", "propValueMask": "", "sortType": "sys_name"},
            "force": 1, "flags": 1, "from": 0, "to": 0
        }),
        "sid": eid
    }
    return requests.post(url, params=params).json()


resources = get_resources(eid)

In [8]:
print(json.dumps(resources, indent=4))  # 👈 pretty output

{
    "searchSpec": {
        "itemsType": "avl_resource",
        "propName": "",
        "propValueMask": "",
        "sortType": "sys_name",
        "propType": "",
        "or_logic": "0"
    },
    "dataFlags": 1,
    "totalItemsCount": 277,
    "indexFrom": 0,
    "indexTo": 0,
    "items": [
        {
            "nm": "20CUBE",
            "cls": 3,
            "id": 26749909,
            "mu": 0,
            "uacl": 17627904754211
        },
        {
            "nm": "Abdi Yusuf",
            "cls": 3,
            "id": 22434436,
            "mu": 0,
            "uacl": 17627904754211
        },
        {
            "nm": "African Rail opportunities",
            "cls": 3,
            "id": 26090802,
            "mu": 0,
            "uacl": 17627904754211
        },
        {
            "nm": "Agility",
            "cls": 3,
            "id": 19310276,
            "mu": 0,
            "uacl": 17627904754211
        },
        {
            "nm": "Alliad",
            "cls"

In [9]:
# Save to file
with open("get_resources.json", "w") as f:
    json.dump(resources, f, indent=4)

print("JSON saved successfully!")

JSON saved successfully!


## Getting all the Templates

In [10]:
def get_all_templates(eid):
    url = "https://hst-api.wialon.com/wialon/ajax.html"

    params = {
        "svc": "core/search_items",
        "params": json.dumps({
            "spec": {
                "itemsType": "avl_resource",
                "propName": "sys_name",
                "propValueMask": "*",   # <-- get ALL resources
                "sortType": "sys_name"
            },
            "force": 1,
            "flags": 8193,   # resource + report templates
            "from": 0,
            "to": 0
        }),
        "sid": eid
    }

    response = requests.post(url, params=params)
    return response.json()

In [11]:
import json

data = get_all_templates(eid)

output = []

for resource in data.get("items", []):
    resource_id = resource.get("id")
    resource_name = resource.get("nm")

    templates = resource.get("rep", {})

    if templates:
        for template_id, template in templates.items():
            output.append({
                "resource_name": resource_name,
                "resource_id": resource_id,
                "template_id": template_id,
                "template_name": template.get("n")
            })
    else:
        output.append({
            "resource_name": resource_name,
            "resource_id": resource_id,
            "template_id": None,
            "template_name": None
        })

In [12]:
print(json.dumps(output, indent=4))

[
    {
        "resource_name": "20CUBE",
        "resource_id": 26749909,
        "template_id": "1",
        "template_name": "20CUBE - service report"
    },
    {
        "resource_name": "20CUBE",
        "resource_id": 26749909,
        "template_id": "2",
        "template_name": "20CUBE - Detailed Event Report per group"
    },
    {
        "resource_name": "20CUBE",
        "resource_id": 26749909,
        "template_id": "3",
        "template_name": "20CUBE - Asset Positions Per Group"
    },
    {
        "resource_name": "20CUBE",
        "resource_id": 26749909,
        "template_id": "4",
        "template_name": "20CUBE - Detailed Event Report per Unit"
    },
    {
        "resource_name": "20CUBE",
        "resource_id": 26749909,
        "template_id": "5",
        "template_name": "20CUBE - detailed journey report per day"
    },
    {
        "resource_name": "20CUBE",
        "resource_id": 26749909,
        "template_id": "6",
        "template_name": "20CUBE - 

In [13]:
# Filter templates for a specific resource
resource_id_to_filter = 29754026  # change to your desired resource ID

filtered = [
    item for item in output
    if item.get("resource_id") == resource_id_to_filter and item.get("template_id") is not None
]

print(json.dumps(filtered, indent=4))

[
    {
        "resource_name": "Food 4 Education",
        "resource_id": 29754026,
        "template_id": "1",
        "template_name": "F4E Fuel Analysis Report (Per Vehicle)"
    },
    {
        "resource_name": "Food 4 Education",
        "resource_id": 29754026,
        "template_id": "2",
        "template_name": "F4E - Driver Behavior Events"
    },
    {
        "resource_name": "Food 4 Education",
        "resource_id": 29754026,
        "template_id": "3",
        "template_name": "F4E - Geofence Report"
    },
    {
        "resource_name": "Food 4 Education",
        "resource_id": 29754026,
        "template_id": "4",
        "template_name": "F4E - Mileage Report"
    },
    {
        "resource_name": "Food 4 Education",
        "resource_id": 29754026,
        "template_id": "5",
        "template_name": "F4E - Speeding Report"
    },
    {
        "resource_name": "Food 4 Education",
        "resource_id": 29754026,
        "template_id": "6",
        "template_name"

In [14]:
# Save to JSON file
filename = "all_resource_templates.json"

with open(filename, "w", encoding="utf-8") as f:
    json.dump(output, f, indent=4, ensure_ascii=False)

print(f"Saved successfully to {filename}")

Saved successfully to all_resource_templates.json


In [15]:
# Short output: resource name, ID, and template count
output = []
for res in data.get("items", []):
    resource_name = res.get("nm")
    resource_id = res.get("id")
    templates = res.get("rep", {}) or {}
    output.append({
        "resource_name": resource_name,
        "resource_id": resource_id,
        "template_count": len(templates)
    })

# Print
for o in output:
    print(f"{o['resource_name']} | ID: {o['resource_id']} | Templates: {o['template_count']}")

20CUBE | ID: 26749909 | Templates: 24
Abdi Yusuf | ID: 22434436 | Templates: 0
African Rail opportunities | ID: 26090802 | Templates: 35
Agility | ID: 19310276 | Templates: 23
Alliad | ID: 28462319 | Templates: 17
AMANI TRANSPORTERS | ID: 27027199 | Templates: 10
Anwarali | ID: 29166211 | Templates: 14
AQAUMIST LTD | ID: 29564956 | Templates: 1
Aquamist | ID: 29561763 | Templates: 10
Asahabito | ID: 17717423 | Templates: 12
AstrumDemo | ID: 28604408 | Templates: 11
Bahari Forwarders | ID: 18971188 | Templates: 20
Bash Contractors | ID: 22777983 | Templates: 12
Bash Hauliers | ID: 22135146 | Templates: 15
Basic920 Resources | ID: 22824415 | Templates: 17
Basra | ID: 22563115 | Templates: 18
Basraa | ID: 22563127 | Templates: 19
BAT-Demo | ID: 27030005 | Templates: 0
BEMONEY | ID: 27924440 | Templates: 18
Bhachu Group | ID: 25739155 | Templates: 22
BINGWA | ID: 27455003 | Templates: 1
Bingwa cathrine | ID: 27527996 | Templates: 0
Bingwa@bingwatransporters.com-SLY | ID: 27514647 | Templat

In [16]:
resource_id_to_filter = 26749909  # change this to the desired resource ID

filtered = [
    item for item in output
    if item["resource_id"] == resource_id_to_filter
]

print(json.dumps(filtered, indent=4))

[
    {
        "resource_name": "20CUBE",
        "resource_id": 26749909,
        "template_count": 24
    }
]


## Getting all Groups and their Units

In [17]:
def get_all_groups(eid):
    avl_groups = 'https://hst-api.wialon.com/wialon/ajax.html?svc=core/search_items&params='\
        '{"spec":'\
            '{"itemsType":"avl_unit_group","propName":"sys_name","propValueMask":"*","sortType":"sys_name"}'\
                ',"force":1,"flags":8193,"from":0,"to":0}&sid=' + eid
    r = requests. post(avl_groups)
    res = json. loads(r.text)
    return res

In [18]:
data = get_all_groups(eid)

for group in data.get("items", []):
    group_id = group.get("id")
    group_name = group.get("nm")
    units = group.get("u", [])

    print(
        f"Group ID: {group_id} | "
        f"Group Name: {group_name} | "
        f"Units Count: {len(units)}"
    )

Group ID: 26749961 | Group Name: 20CUBE | Units Count: 0
Group ID: 29136996 | Group Name: 20Cube  Test | Units Count: 0
Group ID: 29693173 | Group Name: 65 K/HR | Units Count: 26
Group ID: 29693176 | Group Name: 70K/HR | Units Count: 6
Group ID: 29693181 | Group Name: 80 KM/HR | Units Count: 14
Group ID: 29693183 | Group Name: 120 K/HR | Units Count: 7
Group ID: 28212445 | Group Name: Acceler Demo | Units Count: 0
Group ID: 26943025 | Group Name: Accurate | Units Count: 22
Group ID: 29083795 | Group Name: ADMIN, CIVIL & MACHINERY | Units Count: 18
Group ID: 30026549 | Group Name: Africa Global Logistics V8 | Units Count: 34
Group ID: 26351401 | Group Name: Africa Rail | Units Count: 16
Group ID: 26090817 | Group Name: African Rail Opportunities | Units Count: 17
Group ID: 25238031 | Group Name: All Bash | Units Count: 0
Group ID: 27992376 | Group Name: All ETS SABRINA | Units Count: 33
Group ID: 28212584 | Group Name: All Expeditors | Units Count: 51
Group ID: 27309381 | Group Name: Al

In [19]:
clean_output = []

for group in data.get("items", []):
    clean_output.append({
        "group_id": group.get("id"),
        "group_name": group.get("nm"),
        "units_count": len(group.get("u", []))
    })

with open("clean_groups.json", "w", encoding="utf-8") as f:
    json.dump(clean_output, f, indent=4, ensure_ascii=False)

print("Saved clean group data.")

Saved clean group data.


In [20]:
import requests
import json

def get_all_groups(eid):
    url = 'https://hst-api.wialon.com/wialon/ajax.html'
    params = {
        "svc": "core/search_items",
        "params": json.dumps({
            "spec": {
                "itemsType": "avl_unit_group",
                "propName": "sys_name",
                "propValueMask": "*",
                "sortType": "sys_name"
            },
            "force": 1,
            "flags": 8193,  # Include units in the response (IDs)
            "from": 0,
            "to": 0
        }),
        "sid": eid
    }
    r = requests.post(url, data=params)
    return r.json()

def get_units(eid):
    url = "https://hst-api.wialon.com/wialon/ajax.html"
    payload = {
        "svc": "core/search_items",
        "params": json.dumps({
            "spec": {
                "itemsType": "avl_unit",
                "propName": "sys_name",
                "propValueMask": "*",
                "sortType": "sys_name"
            },
            "force": 1,
            "flags": 1,  # minimal flags
            "from": 0,
            "to": 0
        }),
        "sid": eid
    }
    r = requests.post(url, data=payload)
    return r.json()


# Fetch groups and units
groups_data = get_all_groups(eid)
units_data = get_units(eid)

# Build unit ID → unit name map
unit_id_map = {unit['id']: unit['nm'] for unit in units_data.get('items', [])}

# Build clean output
clean_groups = []
for group in groups_data.get("items", []):
    group_id = group.get("id")
    group_name = group.get("nm")
    units_raw = group.get("u", [])
    
    units = []
    for u in units_raw:
        if isinstance(u, dict):
            # sometimes full unit object is in group
            unit_id = u.get("id")
            unit_name = u.get("nm")
        else:
            # u is just an int → lookup in unit map
            unit_id = u
            unit_name = unit_id_map.get(u, "Unknown")
        units.append({"unit_id": unit_id, "unit_name": unit_name})
    
    # Print
    print(f"Group ID: {group_id} | Group Name: {group_name} | Units Count: {len(units)}")
    for u in units:
        print(f"    - {u['unit_name']} (ID: {u['unit_id']})")
    
    clean_groups.append({
        "group_id": group_id,
        "group_name": group_name,
        "units": units
    })

# Save to JSON
with open("groups_with_units.json", "w", encoding="utf-8") as f:
    json.dump(clean_groups, f, indent=4, ensure_ascii=False)

print("Saved groups with units successfully!")

Group ID: 26749961 | Group Name: 20CUBE | Units Count: 0
Group ID: 29136996 | Group Name: 20Cube  Test | Units Count: 0
Group ID: 29693173 | Group Name: 65 K/HR | Units Count: 26
    - AQUAMIST -  KAV 313M (ID: 29561863)
    - AQUAMIST - KCJ 100X (ID: 29561864)
    - AQUAMIST - KCB 532E (ID: 29561869)
    - AQUAMIST - KAV 941S (ID: 29561873)
    - AQUAMIST - KAX 871S (ID: 29561875)
    - AQUAMIST - KBF 933N (ID: 29564828)
    - AQUAMIST - KBS 292D (ID: 29564830)
    - AQUAMIST - KAZ 581G (ID: 29564831)
    - AQUAMIST - KAV 433V (ID: 29564833)
    - AQUAMIST - KAV 274M (ID: 29564837)
    - AQUAMIST - KBN 043A (ID: 29570587)
    - AQUAMIST - KBH 623V (ID: 29578706)
    - AQUAMIST - KBY 427Q (ID: 29578712)
    - AQUAMIST - KBU 224Z (ID: 29579017)
    - AQUAMIST - KBL 204A (ID: 29579019)
    - AQUAMIST - KCL 902M (ID: 29579021)
    - AQUAMIST - KAV 437B (ID: 29579023)
    - AQUAMIST - KCH 773B (ID: 29579024)
    - AQUAMIST - KAT 486Q (ID: 29586956)
    - AQUAMIST - KBD 650P (ID: 29586961)


## Getting all the Units

In [21]:
def get_units(eid):
    url = "https://hst-api.wialon.com/wialon/ajax.html"

    payload = {
        "svc": "core/search_items",
        "params": json.dumps({
            "spec": {
                "itemsType": "avl_unit",
                "propName": "sys_name",
                "propValueMask": "*",
                "sortType": "sys_name"
            },
            "force": 1,
            "flags": 1,
            "from": 0,
            "to": 0
        }),
        "sid": eid
    }

    r = requests.post(url, data=payload)
    return r.json()


In [22]:
get_units(eid)

{'searchSpec': {'itemsType': 'avl_unit',
  'propName': 'sys_name',
  'propValueMask': '*',
  'sortType': 'sys_name',
  'propType': '',
  'or_logic': '0'},
 'dataFlags': 1,
 'totalItemsCount': 4820,
 'indexFrom': 0,
 'indexTo': 0,
 'items': [{'nm': 'Accurate -  KAX 841L PICK-UP',
   'cls': 2,
   'id': 25775908,
   'mu': 0,
   'uacl': 8815420391971},
  {'nm': 'Accurate -  KDD 139E Truck',
   'cls': 2,
   'id': 25775870,
   'mu': 0,
   'uacl': 8815420391971},
  {'nm': 'Accurate -  KDD 643E Truck',
   'cls': 2,
   'id': 25753111,
   'mu': 0,
   'uacl': 8815420391971},
  {'nm': 'Accurate - KAS 742J Truck',
   'cls': 2,
   'id': 25746711,
   'mu': 0,
   'uacl': 8815420391971},
  {'nm': 'Accurate - KAV 283G TRUCK',
   'cls': 2,
   'id': 25746717,
   'mu': 0,
   'uacl': 8815420391971},
  {'nm': 'Accurate - KAV 850P TRUCK',
   'cls': 2,
   'id': 25753138,
   'mu': 0,
   'uacl': 8815420391971},
  {'nm': 'Accurate - KAX 670P  FORD DOUBLE CAB',
   'cls': 2,
   'id': 25739348,
   'mu': 0,
   'uacl'

In [23]:
# Save to file
data = get_units(eid)
with open("get_units.json", "w") as f:
    json.dump(data, f, indent=4)

print("JSON saved successfully!")

JSON saved successfully!


In [24]:
29754675

29754675

## Executing Reports

In [25]:
import pandas as pd
from datetime import datetime, timezone

# Explicit UTC interval: 22 Feb 2026 07:00 → 23 Feb 2026 15:30
start_utc = datetime(2026, 4, 1, 0, 0, tzinfo=timezone.utc)
end_utc = datetime(2026, 4, 15, 15, 30, tzinfo=timezone.utc)

from_ts = int(start_utc.timestamp())
to_ts = int(end_utc.timestamp())

# 1) Execute speeding report for West Kenya (same parameters as Wialon UI)
exec_payload = {
    "svc": "report/exec_report",
    "params": json.dumps(
        {
            "reportResourceId": 17082202,
            "reportTemplateId": 220,
            "reportObjectId": 30182477,
            "reportObjectSecId": 0,
            "interval": {
                "flags": 0,
                "from": from_ts,
                "to": to_ts,
            },
        }
    ),
    "sid": eid,
}

exec_response = requests.post(
    "https://hst-api.wialon.com/wialon/ajax.html",
    data=exec_payload,
)
exec_result = exec_response.json()

# 2) Helper function to fetch any table by index
def fetch_table(report_tables, table_index):
    if table_index >= len(report_tables):
        print(f"Table index {table_index} not found. Only {len(report_tables)} table(s) available.")
        return pd.DataFrame()

    table_meta = report_tables[table_index]
    headers = table_meta.get("header", [])
    row_count = table_meta.get("rows", 0)

    rows_payload = {
        "svc": "report/get_result_rows",
        "params": json.dumps(
            {
                "tableIndex": table_index,   # <-- key change: use the target index
                "indexFrom": 0,
                "indexTo": max(row_count - 1, 0),
            }
        ),
        "sid": eid,
    }

    rows_response = requests.post(
        "https://hst-api.wialon.com/wialon/ajax.html",
        data=rows_payload,
    )
    rows_json = rows_response.json()

    rows_data = []
    for row in rows_json:
        cells = row.get("c", [])
        values = [c.get("t") if isinstance(c, dict) else c for c in cells]
        rows_data.append(values)

    if rows_data:
        max_cols = min(len(headers), len(rows_data[0]))
        return pd.DataFrame(rows_data, columns=headers[:max_cols])
    else:
        return pd.DataFrame(columns=headers)


report_tables = exec_result.get("reportResult", {}).get("tables", [])

# Fetch both sheets
ena_coach_summary  = fetch_table(report_tables, table_index=0)  # Sheet 1
ena_coach_ecodriving = fetch_table(report_tables, table_index=1)  # Sheet 2



In [26]:
ena_coach_summary

,Grouping,Mileage in all messages,Avg. speed,Engine hours,Avg. mileage per unit of fuel by FLS,Total fillings,Total drains,Filled,Drained,Consumed by AbsFCS,Avg. consumption by AbsFCS,Max. value of custom sensor,Max. speed
0,ENA - KDE 181Q - FMC150,12746 km,36 km/h,11 days 18:59:28,2.82 km,17,0,4312 l,0.00 l,4502 l,35 l/100 km,-----,115 km/h
1,ENA COACH - KDE 182Q,12594 km,36 km/h,11 days 6:00:01,2.03 km,107,130,6056 l,3456 l,4219 l,34 l/100 km,-----,136 km/h
2,ENA COACH - KDS 923C,0.00 km,0 km/h,0:00:00,0.00 km,0,0,0.00 l,0.00 l,0.00 l,0.00 l/100 km,-----,0 km/h


In [27]:
ena_coach_ecodriving

,№,Grouping,Violation,Beginning,Initial location,End,Final location,Value,Max. speed,Duration,Mileage,Engine hours duration,Count,Trip duration
0,1,ENA - KDE 181Q - FMC150,,01.04.2026 00:00:12,"Nakuru-Eldoret Rd., Kenya, 1.38 km from Tulwet...",15.04.2026 14:33:54,"Shimo La Tewa Close, Nairobi, Kenya",,115 km/h,16 days 19:44:28,24534 km,11 days 18:59:28,27270,9 days 17:30:28
1,2,ENA COACH - KDE 182Q,,01.04.2026 06:51:19,"Mama Diana Mshega Road, Mombasa, Kenya",15.04.2026 06:28:46,"Kisumu-Busia Road, Busia, Kenya",,136 km/h,13 days 2:51:12,10294 km,11 days 6:00:01,34814,4 days 3:32:31
2,3,ENA COACH - KDS 923C,,-----,,-----,,,0 km/h,0:00:00,0.00 km,0:00:00,0,0:00:00


In [28]:
def detailize_table(table_index, row_index, col_index=0):
    """
    Fetch detailization (subrows) for a specific row in a report table
    """
    detail_payload = {
        "svc": "report/get_result_subrows",
        "params": json.dumps(
            {
                "tableIndex": table_index,
                "rowIndex": row_index,
                "colIndex": col_index,  # usually 0 works, but depends on report structure
                "indexFrom": 0,
                "indexTo": 1000,  # adjust if you expect more rows
            }
        ),
        "sid": eid,
    }

    response = requests.post(
        "https://hst-api.wialon.com/wialon/ajax.html",
        data=detail_payload,
    )

    result = response.json()

    rows_data = []
    for row in result:
        cells = row.get("c", [])
        values = [c.get("t") if isinstance(c, dict) else c for c in cells]
        rows_data.append(values)

    return rows_data

In [ ]:
all_details = []

table_index = 1  # ecodriving table

for i in range(len(ena_coach_ecodriving)):
    details = detailize_table(table_index=table_index, row_index=i)

    for d in details:
        all_details.append([i] + d)  # attach parent row index

# Convert to DataFrame
detail_df = pd.DataFrame(all_details)

In [ ]:
num_cols = detail_df.shape[1]

detail_columns = ["parent_row"] + [f"field_{i}" for i in range(1, num_cols)]

detail_df.columns = detail_columns

In [ ]:
# Drop unwanted columns
detail_df = detail_df.drop(columns=["parent_row", "field_1", "field_12"], errors="ignore")

# Keep only expected number of columns
detail_df = detail_df.iloc[:, :100000]

# Rename columns
detail_df.columns = [
    "Grouping",
    "Violation",
    "Beginning",
    "Initial location",
    "End",
    "Final location",
    "Avg. speed",
    "Max. speed",
    "Duration",
    "Mileage",
    "Count"
]

In [ ]:
detail_df.shape

(92739, 11)

In [ ]:
detail_df.head()

,Grouping,Violation,Beginning,Initial location,End,Final location,Avg. speed,Max. speed,Duration,Mileage,Count
0,ENA COACH - KDE 181Q,Accelerator &gt; 70%,01.04.2026 00:00:14,"Old Nairobi Road, Kenya, Royal B",01.04.2026 00:02:44,"Old Nairobi Road, Kenya, Royal B",88.00,0 km/h,0:02:30,0.00 km,1
1,ENA COACH - KDE 181Q,Engine Stress,01.04.2026 00:00:14,"Old Nairobi Road, Kenya, Royal B",01.04.2026 00:00:44,"Old Nairobi Road, Kenya, Royal B",90.00,0 km/h,0:00:30,0.00 km,1
2,ENA COACH - KDE 181Q,Accelerator &lt; 40 %,01.04.2026 00:00:14,"Old Nairobi Road, Kenya, Royal B",01.04.2026 00:02:44,"Old Nairobi Road, Kenya, Royal B",88.00,0 km/h,0:02:30,0.00 km,1
3,ENA COACH - KDE 181Q,Engine Stress,01.04.2026 00:01:44,"Old Nairobi Road, Kenya, Royal B",01.04.2026 00:02:14,"Old Nairobi Road, Kenya, Royal B",89.00,0 km/h,0:00:30,0.00 km,1
4,ENA COACH - KDE 181Q,Harsh Braking,01.04.2026 00:02:56,"Old Nairobi Road, Kenya, Royal B",01.04.2026 00:03:00,"Old Nairobi Road, Kenya, Royal B",1.00,0 km/h,0:00:04,0.00 km,1


In [ ]:
detail_df.tail()

,Grouping,Violation,Beginning,Initial location,End,Final location,Avg. speed,Max. speed,Duration,Mileage,Count
92734,ENA COACH - KDE 182Q,Accelerator &lt; 40 %,15.04.2026 06:05:26,"Kisumu-Busia Road, Busia, Kenya",15.04.2026 06:05:34,"Kisumu-Busia Road, Busia, Kenya",51.00,49 km/h,0:00:08,0.09 km,1
92735,ENA COACH - KDE 182Q,Green Band Driving,15.04.2026 06:05:26,"Kisumu-Busia Road, Busia, Kenya",15.04.2026 06:05:35,"Kisumu-Busia Road, Busia, Kenya",1078.00,49 km/h,0:00:09,0.12 km,1
92736,ENA COACH - KDE 182Q,Green Band Driving,15.04.2026 06:05:44,"Kisumu-Busia Road, Busia, Kenya",15.04.2026 06:06:04,"Kisumu-Busia Road, Busia, Kenya",1066.00,28 km/h,0:00:20,0.14 km,1
92737,ENA COACH - KDE 182Q,Harsh Braking,15.04.2026 06:28:24,"Kisumu-Busia Road, Busia, Kenya",15.04.2026 06:28:46,"Kisumu-Busia Road, Busia, Kenya",1.00,0 km/h,0:00:22,0.00 km,1
92738,ENA COACH - KDS 923C,,-----,,-----,,,0 km/h,0:00:00,0.00 km,0


In [ ]:
def exec_report_for_group(eid, resource_id, template_id, group_id):
    payload = {
        "svc": "report/exec_report",
        "params": json.dumps({
            "reportResourceId": resource_id,
            "reportTemplateId": template_id,
            "reportObjectId": group_id,
            "reportObjectSecId": 0,
            "interval": {
                "flags": 1025,
                "from": 1735689600,
                "to": int(time.time())
            }
        }),
        "sid": eid
    }

    response = requests.post(
        "https://hst-api.wialon.com/wialon/ajax.html",
        data=payload
    )

    return response.json()


In [ ]:
#exec_report_for_group(eid, 25601229, 26, 29820640)

In [ ]:
exec_report_for_group(eid, 29754026, 3, 29754675)

{'reportResult': {'msgsRendered': 0,
  'stats': [],
  'tables': [{'name': 'unit_group_zones_visit',
    'label': 'Geofences',
    'grouping': {'type': 'unit'},
    'flags': 256,
    'rows': 3,
    'level': 2,
    'columns': 5,
    'header': ['Grouping', 'Geofence', 'Time in', 'Time out', 'Duration in'],
    'header_type': ['',
     'zone_name',
     'time_begin',
     'time_end',
     'duration_in']}],
  'attachments': []}}

In [ ]:
def get_report_rows(eid, table_index=0, index_from=0, index_to=1000):
    payload = {
        "svc": "report/get_result_rows",
        "params": json.dumps({
            "tableIndex": table_index,
            "indexFrom": index_from,
            "indexTo": index_to
        }),
        "sid": eid
    }

    response = requests.post(
        "https://hst-api.wialon.com/wialon/ajax.html",
        data=payload
    )

    return response.json()

In [ ]:
# 1️⃣ Execute report
exec_report_for_group(eid, 29754026, 3, 29754675)

# 2️⃣ Get actual rows
rows = get_report_rows(eid)

print(rows)

[{'n': 0, 'i1': 128, 'i2': 129178, 't1': 1761222422, 't2': 1775583263, 'd': 1393, 'mrk': 0, 'c': ['F4E - KCW 287X', '', {'t': '2025-10-23 12:27:02', 'v': 1761222422, 'y': -1.3069116, 'x': 36.8474783, 'u': 29754685}, {'t': '2026-04-07 17:34:23', 'v': 1775583263, 'y': -1.2790916, 'x': 36.8232516, 'u': 29754685}, '84 days 17:10:58']}, {'n': 1, 'i1': 31, 'i2': 109846, 't1': 1761222233, 't2': 1777459136, 'd': 1181, 'mrk': 0, 'c': ['F4E - KCY 152C', '', {'t': '2025-10-23 12:23:53', 'v': 1761222233, 'y': -1.306915, 'x': 36.8474983, 'u': 29754686}, {'t': '2026-04-29 10:38:56', 'v': 1777459136, 'y': -1.1633516, 'x': 36.8483566, 'u': 29754686}, '101 days 4:36:53']}, {'n': 2, 'i1': 58, 'i2': 137737, 't1': 1761123196, 't2': 1777452979, 'd': 1432, 'mrk': 0, 'c': ['F4E - KCY 806C', '', {'t': '2025-10-22 08:53:16', 'v': 1761123196, 'y': -1.3069916, 'x': 36.8476233, 'u': 29754683}, {'t': '2026-04-29 08:56:19', 'v': 1777452979, 'y': -1.0819583, 'x': 37.011585, 'u': 29754683}, '46 days 20:57:05']}]


In [ ]:
import requests
import json
import time
import pandas as pd

# 1️⃣ Execute Report
def exec_report_for_group(eid, resource_id, template_id, group_id):
    payload = {
        "svc": "report/exec_report",
        "params": json.dumps({
            "reportResourceId": resource_id,
            "reportTemplateId": template_id,
            "reportObjectId": group_id,
            "reportObjectSecId": 0,
            "interval": {
                "flags": 1025,
                "from": 1735689600,   # Jan 1 2025
                "to": int(time.time())
            }
        }),
        "sid": eid
    }

    requests.post(
        "https://hst-api.wialon.com/wialon/ajax.html",
        data=payload
    )


# 2️⃣ Get Report Rows
def get_report_rows(eid):
    payload = {
        "svc": "report/get_result_rows",
        "params": json.dumps({
            "tableIndex": 0,
            "indexFrom": 0,
            "indexTo": 10000
        }),
        "sid": eid
    }

    response = requests.post(
        "https://hst-api.wialon.com/wialon/ajax.html",
        data=payload
    )

    return response.json()


# Execute report
exec_report_for_group(eid, 25601229, 26, 29820640)

# Get rows
rows = get_report_rows(eid)

# 🔎 Check if response contains error
if isinstance(rows, dict) and "error" in rows:
    print("API Error:", rows)
else:
    clean_data = []

    for row in rows:   # rows must be a list
        if isinstance(row, dict) and "c" in row:
            c = row["c"]
            clean_data.append([
                c[0],
                c[1],
                c[2]["t"],
                c[3]["t"],
                c[4]
            ])

    df = pd.DataFrame(
        clean_data
    )

    df.to_excel("wialon_report.xlsx", index=False)

    print("Excel file created successfully ✅")

Excel file created successfully ✅


In [ ]:
# import time

# exec_report_for_group(eid, 29754026, 3, 29754675)

# time.sleep(2)   # wait 2 seconds before fetching rows

# rows = get_report_rows(eid)

In [ ]:
# def exec_report_units_with_registration(eid, resource_id, template_id, group_id):
#     """Run exec_report_for_group and return a DataFrame of units
#     with registration name shown before unit ID.
#     """
#     # Run the report
#     report_data = exec_report_for_group(eid, resource_id, template_id, group_id)

#     # Build unit ID -> registration name map
#     units_data = get_units(eid)
#     unit_id_map = {unit["id"]: unit["nm"] for unit in units_data.get("items", [])}

#     # Extract units from report layer
#     units = report_data.get("reportLayer", {}).get("units", [])

#     rows = []
#     for u in units:
#         unit_id = u.get("id")
#         registration_name = unit_id_map.get(unit_id, "Unknown")
#         rows.append(
#             {
#                 "registration_name": registration_name,
#                 "unit_id": unit_id,
#                 "mileage": u.get("mileage"),
#                 "max_speed": u.get("max_speed"),
#             }
#         )

#     # Ensure registration name is before unit ID in the final view
#     return pd.DataFrame(rows, columns=["registration_name", "unit_id", "mileage", "max_speed"])

In [ ]:
# Example: same parameters as previous exec_report_for_group call
# exec_units_df = exec_report_units_with_registration(eid, 25601229, 13, 29090593)
# exec_units_df

In [ ]:
        # "resource_name": "WEST KENYA SUGAR",
        # "resource_id": 28842931,
        # "template_id": "3",
        # "template_name": "West Kenya - Speeding Report"

# Time should be from yesterday till today

#         "group_id": 28811648,
#         "group_name": "All West Kenya Sugar vehicles",
#         "units_count": 377

In [ ]:
#exec_report_for_group(eid, 25601229, 26, 29820640) Menengai

In [ ]:
import pandas as pd
from datetime import datetime, timezone

# Explicit UTC interval: 22 Feb 2026 07:00 → 23 Feb 2026 15:30
start_utc = datetime(2026, 2, 22, 7, 0, tzinfo=timezone.utc)
end_utc = datetime(2026, 2, 23, 15, 30, tzinfo=timezone.utc)

from_ts = int(start_utc.timestamp())
to_ts = int(end_utc.timestamp())

# 1) Execute speeding report for West Kenya (same parameters as Wialon UI)
exec_payload = {
    "svc": "report/exec_report",
    "params": json.dumps(
        {
            "reportResourceId": 25601229,  # WEST KENYA SUGAR
            "reportTemplateId": 26,         # West Kenya - Speeding Report
            "reportObjectId": 29820640,    # All West Kenya Sugar vehicles group
            "reportObjectSecId": 0,
            "interval": {
                "flags": 0,
                "from": from_ts,
                "to": to_ts,
            },
        }
    ),
    "sid": eid,
}

exec_response = requests.post(
    "https://hst-api.wialon.com/wialon/ajax.html",
    data=exec_payload,
)
exec_result = exec_response.json()

# 2) Read table meta to know how many rows to fetch
report_tables = exec_result.get("reportResult", {}).get("tables", [])
if not report_tables:
    west_kenya_speeding_df = pd.DataFrame()
else:
    table_meta = report_tables[0]
    headers = table_meta.get("header", [])
    row_count = table_meta.get("rows", 0)

    # 3) Call report/get_result_rows to fetch all rows
    rows_payload = {
        "svc": "report/get_result_rows",
        "params": json.dumps(
            {
                "tableIndex": 0,          # first (and only) table
                "indexFrom": 0,
                "indexTo": max(row_count - 1, 0),
            }
        ),
        "sid": eid,
    }

    rows_response = requests.post(
        "https://hst-api.wialon.com/wialon/ajax.html",
        data=rows_payload,
    )
    rows_json = rows_response.json()

    rows_data = []
    for row in rows_json:
        cells = row.get("c", [])
        values = []
        for c in cells:
            if isinstance(c, dict):
                values.append(c.get("t"))  # formatted text value
            else:
                values.append(c)
        rows_data.append(values)

    if rows_data:
        max_cols = min(len(headers), len(rows_data[0]))
        west_kenya_speeding_df = pd.DataFrame(rows_data, columns=headers[:max_cols])
    else:
        west_kenya_speeding_df = pd.DataFrame(columns=headers)

west_kenya_speeding_df

,Grouping,Last message time,Last coordinates time,Location,Speed
0,ADMIN - Menengai - KBV 789M,29.04.2026 10:34:23,29.04.2026 10:34:23,"Nakuru-Kisumu Road, Nakuru, Kenya",0 km/h
1,ADMIN - Menengai - KCM 234J,29.04.2026 10:45:22,29.04.2026 10:45:22,"Nakuru-Kisumu Road, Nakuru, Kenya",0 km/h
2,CIVIL - Menengai - EXCAVATOR 002,29.04.2026 10:23:01,29.04.2026 10:23:01,"Oduro, Kenya",0 km/h
3,CIVIL - Menengai - EXCAVATOR001,29.04.2026 10:39:21,29.04.2026 10:39:21,"D267, Kenya, 2.91 km from Mwonyonyi",0 km/h
4,CIVIL - Menengai - KBV 487K,03.10.2025 12:23:29,03.10.2025 12:23:29,"D267, Kenya, 1.91 km from Kambi-Mwanza",0 km/h
...,...,...,...,...,...
221,VS - Menengai -KDE 136E,29.04.2026 10:43:44,29.04.2026 10:43:44,"Pate Road, Nairobi, Kenya",0 km/h
222,VS - Menengai -KDE 138E,29.04.2026 10:33:41,29.04.2026 10:33:41,"Nakuru-Kisumu Road, Nakuru, Kenya",0 km/h
223,VS - Menengai -KDE 140E,29.04.2026 10:47:51,29.04.2026 10:47:51,"C70, Kenya, Kandara",49 km/h
224,VS - Menengai -KDE 144E,29.04.2026 10:47:45,29.04.2026 10:47:45,"Nakuru-Kisumu Road, Nakuru, Kenya",0 km/h


# The Manhattan Project

In [ ]:
        # "resource_name": "Mene-ngai",
        # "resource_id": 25601229,
        # "template_id": "8",
        # "template_name": "Menengai1-Advanced Fuel Analysis Report (Per Vehicle)"

# start_utc = datetime(2026, 2, 22, 7, 0, tzinfo=timezone.utc)
# end_utc = datetime(2026, 2, 23, 15, 30, tzinfo=timezone.utc)

                # "unit_id": 29579024,
                # "unit_name": "AQUAMIST - KCH 773B"

In [ ]:
import pandas as pd
from datetime import datetime, timezone

# Explicit UTC interval: 22 Feb 2026 07:00 → 23 Feb 2026 15:30
start_utc = datetime(2026, 4, 10, 7, 0, tzinfo=timezone.utc)
end_utc = datetime(2026, 4, 10, 15, 30, tzinfo=timezone.utc)

from_ts = int(start_utc.timestamp())
to_ts = int(end_utc.timestamp())

# 1) Execute speeding report for West Kenya (same parameters as Wialon UI)
exec_payload = {
    "svc": "report/exec_report",
    "params": json.dumps(
        {
            "reportResourceId": 17082202,
            "reportTemplateId": 220,
            "reportObjectId": 30182477,
            "reportObjectSecId": 0,
            "interval": {
                "flags": 0,
                "from": from_ts,
                "to": to_ts,
            },
        }
    ),
    "sid": eid,
}

exec_response = requests.post(
    "https://hst-api.wialon.com/wialon/ajax.html",
    data=exec_payload,
)
exec_result = exec_response.json()

# 2) Helper function to fetch any table by index
def fetch_table(report_tables, table_index):
    if table_index >= len(report_tables):
        print(f"Table index {table_index} not found. Only {len(report_tables)} table(s) available.")
        return pd.DataFrame()

    table_meta = report_tables[table_index]
    headers = table_meta.get("header", [])
    row_count = table_meta.get("rows", 0)

    rows_payload = {
        "svc": "report/get_result_rows",
        "params": json.dumps(
            {
                "tableIndex": table_index,   # <-- key change: use the target index
                "indexFrom": 0,
                "indexTo": max(row_count - 1, 0),
            }
        ),
        "sid": eid,
    }

    rows_response = requests.post(
        "https://hst-api.wialon.com/wialon/ajax.html",
        data=rows_payload,
    )
    rows_json = rows_response.json()

    rows_data = []
    for row in rows_json:
        cells = row.get("c", [])
        values = [c.get("t") if isinstance(c, dict) else c for c in cells]
        rows_data.append(values)

    if rows_data:
        max_cols = min(len(headers), len(rows_data[0]))
        return pd.DataFrame(rows_data, columns=headers[:max_cols])
    else:
        return pd.DataFrame(columns=headers)


report_tables = exec_result.get("reportResult", {}).get("tables", [])

# Fetch both sheets
ena_coach_summary  = fetch_table(report_tables, table_index=0)  # Sheet 1
ena_coach_ecodriving = fetch_table(report_tables, table_index=1)  # Sheet 2



In [ ]:
import pandas as pd
from datetime import datetime, timezone

# Explicit UTC interval for Menengai fuel analysis
start_utc = datetime(2026, 4, 10, 7, 0, tzinfo=timezone.utc)
end_utc = datetime(2026, 4, 10, 15, 30, tzinfo=timezone.utc)

from_ts = int(start_utc.timestamp())
to_ts = int(end_utc.timestamp())

# Execute "Menengai1-Advanced Fuel Analysis Report (Per Vehicle)" for AQUAMIST - KCH 773B
exec_payload = {
    "svc": "report/exec_report",
    "params": json.dumps(
        {
            "reportResourceId": 17082202,
            "reportTemplateId": 220,
            "reportObjectId": 30182477,
            "reportObjectSecId": 0,
            "interval": {
                "flags": 0,
                "from": from_ts,
                "to": to_ts,
            },
        }
    ),
    "sid": eid,  # assumes you already have a valid Wialon session ID in `eid`
}

exec_response = requests.post(
    "https://hst-api.wialon.com/wialon/ajax.html",
    data=exec_payload,
)
exec_result = exec_response.json()

# Top level tables (will show the Total row only for this template)
report_tables = exec_result.get("reportResult", {}).get("tables", [])

if not report_tables:
    menengai_fuel_df = pd.DataFrame()
else:
    table_meta = report_tables[0]  # unit_trips / FLS - Fuel Report
    headers = table_meta.get("header", [])
    row_count = table_meta.get("rows", 0)

    print(
        f"Table 0: name={table_meta.get('name')} | label={table_meta.get('label')} | rows={row_count}"
    )

    # The detailed trips/segments are stored as nested rows under the
    # first (Total) row. Use report/get_result_subrows to fetch them.
    subrows_payload = {
        "svc": "report/get_result_subrows",
        "params": json.dumps(
            {
                "tableIndex": 0,
                "rowIndex": 0,  # Total row
            }
        ),
        "sid": eid,
    }

    subrows_response = requests.post(
        "https://hst-api.wialon.com/wialon/ajax.html",
        data=subrows_payload,
    )
    subrows_json = subrows_response.json()

    rows_data = []

    # subrows_json is either a list of nested rows or a dict with {"error": 0}
    if isinstance(subrows_json, list):
        for row in subrows_json:
            cells = row.get("c", [])
            values = []
            for c in cells:
                if isinstance(c, dict):
                    values.append(c.get("t"))  # formatted text value
                else:
                    values.append(c)
            rows_data.append(values)

    # If we didn't get any nested rows, fall back to the single Total row
    if rows_data:
        max_cols = min(len(headers), len(rows_data[0]))
        menengai_fuel_df = pd.DataFrame(rows_data, columns=headers[:max_cols])
    else:
        # Fallback: get the top-level Total row as before
        rows_payload = {
            "svc": "report/get_result_rows",
            "params": json.dumps(
                {
                    "tableIndex": 0,
                    "indexFrom": 0,
                    "indexTo": max(row_count - 1, 0),
                }
            ),
            "sid": eid,
        }

        rows_response = requests.post(
            "https://hst-api.wialon.com/wialon/ajax.html",
            data=rows_payload,
        )
        rows_json = rows_response.json()

        for row in rows_json:
            cells = row.get("c", [])
            values = []
            for c in cells:
                if isinstance(c, dict):
                    values.append(c.get("t"))
                else:
                    values.append(c)
            rows_data.append(values)

        if rows_data:
            max_cols = min(len(headers), len(rows_data[0]))
            menengai_fuel_df = pd.DataFrame(rows_data, columns=headers[:max_cols])
        else:
            menengai_fuel_df = pd.DataFrame(columns=headers)

menengai_fuel_df

Table 0: name=unit_group_generic | label=Summary | rows=3


,Grouping,Mileage in all messages,Avg. speed,Engine hours,Avg. mileage per unit of fuel by FLS,Total fillings,Total drains,Filled,Drained,Consumed by AbsFCS,Avg. consumption by AbsFCS,Max. value of custom sensor,Max. speed
0,ENA COACH - KDE 181Q,78 km,9 km/h,1:41:53,2.45 km,0,0,0.00 l,0.00 l,24 l,31 l/100 km,-----,82 km/h
1,ENA COACH - KDE 182Q,282 km,33 km/h,6:45:17,2.35 km,3,3,200 l,80 l,91 l,32 l/100 km,-----,90 km/h
2,ENA COACH - KDS 923C,0.00 km,0 km/h,0:00:00,0.00 km,0,0,0.00 l,0.00 l,0.00 l,0.00 l/100 km,-----,0 km/h


## Tacho Messages Report

In [ ]:
import pandas as pd
import requests
import json
from datetime import datetime, timezone

# --- DATE ---
start_utc = datetime(2026, 4, 11, 0, 0, tzinfo=timezone.utc)
end_utc   = datetime(2026, 4, 11, 23, 59, tzinfo=timezone.utc)
from_ts = int(start_utc.timestamp())
to_ts   = int(end_utc.timestamp())

# --- LOAD UNIT NAME FROM get_units.json ---
with open("get_units.json", "r") as f:
    units_data = json.load(f)

# ✅ FIX: The JSON has units nested under "items" key, not at the top level
units_list = units_data.get("items", [])

UNIT_ID = 27251184  # your reportObjectId

unit_name = f"Unit {UNIT_ID}"  # fallback
for unit in units_list:
    if unit.get("id") == UNIT_ID:
        unit_name = unit.get("nm", unit_name)
        break

print(f"Unit Name: {unit_name}")

# --- EXECUTE REPORT ---
exec_payload = {
    "svc": "report/exec_report",
    "params": json.dumps({
        "reportResourceId": 26986544,
        "reportTemplateId": 7,
        "reportObjectId": UNIT_ID,
        "reportObjectSecId": 0,
        "interval": {
            "flags": 0,
            "from": from_ts,
            "to": to_ts,
        },
    }),
    "sid": eid,
}
exec_response = requests.post(
    "https://hst-api.wialon.com/wialon/ajax.html",
    data=exec_payload,
)
exec_result = exec_response.json()

# --- FETCH TABLE FUNCTION ---
def fetch_table(report_tables, table_index):
    table_meta = report_tables[table_index]
    headers = table_meta.get("header", [])
    row_count = table_meta.get("rows", 0)
    rows_payload = {
        "svc": "report/get_result_rows",
        "params": json.dumps({
            "tableIndex": table_index,
            "indexFrom": 0,
            "indexTo": max(row_count - 1, 0),
        }),
        "sid": eid,
    }
    rows_response = requests.post(
        "https://hst-api.wialon.com/wialon/ajax.html",
        data=rows_payload,
    )
    rows_json = rows_response.json()
    rows_data = []
    for row in rows_json:
        cells = row.get("c", [])
        values = [c.get("t") if isinstance(c, dict) else c for c in cells]
        rows_data.append(values)
    return pd.DataFrame(rows_data, columns=headers[:len(rows_data[0])] if rows_data else headers)

# --- GET TABLES ---
report_tables = exec_result.get("reportResult", {}).get("tables", [])

for i, t in enumerate(report_tables):
    print(f"{i}: {t.get('name')}")

# --- FIND MESSAGE TRACING TABLE ---
message_table_index = None
for i, t in enumerate(report_tables):
    if "message" in t.get("name", "").lower():
        message_table_index = i
        break

if message_table_index is None:
    raise Exception("Message Tracing table not found!")

# --- FETCH MESSAGE TRACING ---
message_tracing_df = fetch_table(report_tables, message_table_index)

# --- RENAME COLUMNS ---
final_df = message_tracing_df.copy()

final_df = final_df.rename(columns={
    "unit_name": "Unit Name",
    "io_270":    "Fuel Level",
    "io_240":    "Ignition",
    # ✅ io_24 ("Speed") REMOVED — keeping the native speed column only
    "io_16":     "Mileage (Km)",
    "driver":    "Driver ID",
    "drv":       "Driver ID",
})

# --- INJECT EXACT UNIT NAME FROM get_units.json ---
final_df["Unit Name"] = unit_name

# --- CREATE COORDINATES COLUMN ---
if "lat" in final_df.columns and "lon" in final_df.columns:
    final_df["Coordinates"] = final_df["lat"].astype(str) + ", " + final_df["lon"].astype(str)

# --- CONVERT MILEAGE FROM METERS → KM ---
if "Mileage (Km)" in final_df.columns:
    final_df["Mileage (Km)"] = pd.to_numeric(final_df["Mileage (Km)"], errors="coerce") / 1000

# Rename native speed column to title case for display
if "speed" in final_df.columns:
    final_df = final_df.rename(columns={"speed": "Speed"})

columns_needed = [
    "Unit Name",
    "Time",
    "Fuel Level",
    "Ignition",
    "Speed",        # ✅ after Ignition
    "Coordinates",
    "Mileage (Km)",
]

columns_existing = [col for col in columns_needed if col in final_df.columns]
final_df = final_df[columns_existing]

final_df.head(10)

Unit Name: FG - Menengai - KCP 042S
0: unit_messages_tracing
1: unit_chronology


,Unit Name,Time,Fuel Level,Ignition,Speed,Coordinates,Mileage (Km)
0,FG - Menengai - KCP 042S,11.04.2026 00:13:52,545,0.00,0 km/h,"-0.293022, 36.045775",215854.307
1,FG - Menengai - KCP 042S,11.04.2026 00:28:54,545,0.00,0 km/h,"-0.293022, 36.045775",215854.307
2,FG - Menengai - KCP 042S,11.04.2026 00:43:55,545,0.00,0 km/h,"-0.293022, 36.045775",215854.307
3,FG - Menengai - KCP 042S,11.04.2026 00:58:57,545,0.00,0 km/h,"-0.293022, 36.045775",215854.307
4,FG - Menengai - KCP 042S,11.04.2026 01:13:59,545,0.00,0 km/h,"-0.293022, 36.045775",215854.307
5,FG - Menengai - KCP 042S,11.04.2026 01:29:00,545,0.00,0 km/h,"-0.293022, 36.045775",215854.307
6,FG - Menengai - KCP 042S,11.04.2026 01:44:01,544,0.00,0 km/h,"-0.293022, 36.045775",215854.307
7,FG - Menengai - KCP 042S,11.04.2026 01:59:04,544,0.00,0 km/h,"-0.293022, 36.045775",215854.307
8,FG - Menengai - KCP 042S,11.04.2026 02:14:05,543,0.00,0 km/h,"-0.293022, 36.045775",215854.307
9,FG - Menengai - KCP 042S,11.04.2026 02:29:06,543,0.00,0 km/h,"-0.293022, 36.045775",215854.307


In [ ]:
final_df.tail(10)

,Unit Name,Time,Fuel Level,Ignition,Speed,Coordinates,Mileage (Km)
215,FG - Menengai - KCP 042S,11.04.2026 21:36:01,1936,0.00,0 km/h,"-0.288798, 36.044653",215855.571
216,FG - Menengai - KCP 042S,11.04.2026 21:51:03,1936,0.00,0 km/h,"-0.288798, 36.044653",215855.571
217,FG - Menengai - KCP 042S,11.04.2026 22:06:04,1936,0.00,0 km/h,"-0.288798, 36.044653",215855.571
218,FG - Menengai - KCP 042S,11.04.2026 22:21:06,1935,0.00,0 km/h,"-0.288798, 36.044653",215855.571
219,FG - Menengai - KCP 042S,11.04.2026 22:36:07,1935,0.00,0 km/h,"-0.288798, 36.044653",215855.571
220,FG - Menengai - KCP 042S,11.04.2026 22:51:09,1935,0.00,0 km/h,"-0.288798, 36.044653",215855.571
221,FG - Menengai - KCP 042S,11.04.2026 23:06:11,1934,0.00,0 km/h,"-0.288798, 36.044653",215855.571
222,FG - Menengai - KCP 042S,11.04.2026 23:21:12,1934,0.00,0 km/h,"-0.288798, 36.044653",215855.571
223,FG - Menengai - KCP 042S,11.04.2026 23:36:13,1934,0.00,0 km/h,"-0.288798, 36.044653",215855.571
224,FG - Menengai - KCP 042S,11.04.2026 23:51:15,1934,0.00,0 km/h,"-0.288798, 36.044653",215855.571


In [ ]:
with pd.ExcelWriter("Message_Tracing_Report.xlsx", engine="openpyxl") as writer:
    final_df.to_excel(writer, index=False, sheet_name="Message Tracing")
    
    worksheet = writer.sheets["Message Tracing"]
    
    # Auto-fit column widths
    for col in worksheet.columns:
        max_len = max(len(str(cell.value)) if cell.value else 0 for cell in col)
        worksheet.column_dimensions[col[0].column_letter].width = max_len + 2
    
    # Freeze header row
    worksheet.freeze_panes = "A2"

print(f"Done! {len(final_df)} rows exported to message_tracing_report.xlsx")

Done! 225 rows exported to message_tracing_report.xlsx
